# Ablation study: evaluate the released checkpoints

Evaluates the six released checkpoints in `checkpoints/` on the test set without training.
The reported numbers (`released_results/ablation_summary_seed163.csv`) come from GPU training runs; CPU inference can differ by a few windows.

In [1]:
import os, sys, json
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve() / "common"))
import core as C
import numpy as np, pandas as pd, torch
print("torch", torch.__version__, "| device", C.DEVICE)

torch 2.8.0+cu129 | device cuda


In [2]:
@torch.no_grad()
def predict(net, X, bs=2048):
    net.eval()
    X = torch.from_numpy(X)
    return torch.cat([net(X[i:i + bs].to(C.DEVICE)).argmax(1).cpu() for i in range(0, len(X), bs)]).numpy()


def build(mode, S, S11):
    spec = C.ABL[mode]
    Sx = S if spec["data"] == "full" else S11
    nf = len(Sx["feats"])
    if spec["p1"] is None:
        net = C.PureBaselineMLP(nf, 4, spec["hidden"])
    else:
        net = C.DownstreamClassifier(C.make_encoder(spec["enc"], nf), 4, freeze_encoder=True)
    return net.to(C.DEVICE), Sx

In [3]:
ORDER = [("Baseline", "M0"), ("With M1", "M1"), ("With M3", "M3"),
         ("With M1+M2", "M2"), ("With M1+M3", "M4"), ("Proposed", "M5")]
S, S11 = C.prepare("../data/features_21d", "../data/features_11d", verbose=False)
ref = pd.read_csv("released_results/ablation_summary_seed163.csv").set_index("run_id")
rows = []
for label, mode in ORDER:
    net, Sx = build(mode, S, S11)
    net.load_state_dict(torch.load("checkpoints/best_p2_" + mode + "_seed163.pth", map_location="cpu"))
    pred = predict(net, Sx["X_test"])
    rows.append({"configuration": label, "test_correct": int((pred == Sx["y_test"]).sum()),
                 "test_acc_%": round(100 * float((pred == Sx["y_test"]).mean()), 4),
                 "released_correct": int(ref.loc[mode, "test_correct"])})
pd.DataFrame(rows)

[OK]   self-test: one value changed by 1e-9 -> rejected (test: feature digest mismatch)
[OK]   self-test: columns reordered -> rejected (test: column names/order mismatch.)


<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>configuration</th>
      <th>test_correct</th>
      <th>test_acc_%</th>
      <th>released_correct</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>Baseline</td>
      <td>52967</td>
      <td>85.4279</td>
      <td>52962</td>
    </tr>
    <tr>
      <th>1</th>
      <td>With M1</td>
      <td>54294</td>
      <td>87.5681</td>
      <td>54291</td>
    </tr>
    <tr>
      <th>2</th>
      <td>With M3</td>
      <td>54325</td>
      <td>87.6181</td>
      <td>54325</td>
    </tr>
    <tr>
      <th>3</th>
      <td>With M1+M2</td>
      <td>61392</td>
      <td>99.0162</td>
      <td>61389</td>
    </tr>
    <tr>
      <th>4